# Notebook 02: Reasoning-Elicitation Techniques

Companion to Module 02. Real experiments against live `gpt-4o-mini`:
1. Direct-answer vs. Chain-of-Thought — real accuracy, real tokens, real latency together, no assumed winner.
2. Self-consistency — real empirical majority-vote accuracy at k=1/3/5 (from real, non-overlapping sample groups) vs. Module 02's theoretical binomial formula.
3. Real k-sample latency multiplier — real wall-clock sequential vs. parallel timing.

In [1]:
import os
import re
import time
import math
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv())
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
MODEL = "gpt-4o-mini"
print(f"OpenAI client ready. Model: {MODEL}")

OpenAI client ready. Model: gpt-4o-mini


## 1. Direct-Answer vs. Chain-of-Thought: Accuracy, Tokens & Latency Together

5 real multi-step word problems with known correct numeric answers. No assumption CoT wins -- the notebook reports whatever the real trade-off turns out to be.

In [2]:
EVAL_PROBLEMS = [
    ("A bakery baked 8 trays of cookies, with 24 cookies per tray. They set aside 12 cookies for a taste test, then packed the rest into boxes of 15. How many full boxes did they pack?", 12.0),
    ("A train travels 60 miles in the first hour, then increases its speed by 15 miles per hour for the next 2 hours. How many total miles does it travel in the 3 hours?", 210.0),
    ("Sarah has $85. She spends 40% on a jacket, then spends $18 on shoes from what remains. How much money does she have left?", 33.0),
    ("A conference room seats 18 people per row and has 7 rows. If 94 people register but 15 cancel, how many empty seats will there be?", 47.0),
    ("A recipe requires 3/4 cup of sugar for 12 cookies. If you want to make 44 cookies, and each cup of sugar costs $0.80, what is the total cost of sugar needed in dollars (round to the nearest cent)?", 2.20),
]

DIRECT_SYSTEM = "Solve the math word problem. Respond with ONLY the final numeric answer (a number), nothing else."
COT_SYSTEM = "Solve the math word problem step by step, showing your reasoning. On the FINAL line, write exactly: Final Answer: <number>"

def parse_number(text):
    matches = re.findall(r'-?\d+\.?\d*', text.replace(',', ''))
    return float(matches[-1]) if matches else None

def solve_direct(problem_text):
    start = time.perf_counter()
    resp = client.chat.completions.create(
        model=MODEL, temperature=0.0, max_tokens=20,
        messages=[{"role": "system", "content": DIRECT_SYSTEM}, {"role": "user", "content": problem_text}],
    )
    latency_ms = (time.perf_counter() - start) * 1000
    answer = parse_number(resp.choices[0].message.content)
    return answer, latency_ms, resp.usage.total_tokens

def solve_cot(problem_text):
    start = time.perf_counter()
    resp = client.chat.completions.create(
        model=MODEL, temperature=0.0, max_tokens=400,
        messages=[{"role": "system", "content": COT_SYSTEM}, {"role": "user", "content": problem_text}],
    )
    latency_ms = (time.perf_counter() - start) * 1000
    content = resp.choices[0].message.content
    final_line = content.split("Final Answer:")[-1] if "Final Answer:" in content else content
    answer = parse_number(final_line)
    return answer, latency_ms, resp.usage.total_tokens

def run_condition(solve_fn):
    correct, total_tokens, total_latency = 0, 0, 0.0
    results = []
    for problem_text, true_answer in EVAL_PROBLEMS:
        answer, latency_ms, tokens = solve_fn(problem_text)
        is_correct = answer is not None and abs(answer - true_answer) < 0.01
        correct += int(is_correct)
        total_tokens += tokens
        total_latency += latency_ms
        results.append((true_answer, answer, is_correct))
    return correct / len(EVAL_PROBLEMS), total_tokens, total_latency, results

direct_acc, direct_tokens, direct_latency, direct_results = run_condition(solve_direct)
cot_acc, cot_tokens, cot_latency, cot_results = run_condition(solve_cot)

print("=== DIRECT ANSWER ===")
print(f"Accuracy: {direct_acc:.2f} ({int(direct_acc*5)}/5)  Tokens: {direct_tokens}  Latency: {direct_latency:.1f}ms")
for true_a, pred_a, ok in direct_results:
    print(f"  [{'OK' if ok else 'WRONG'}] true={true_a} pred={pred_a}")

print("\n=== CHAIN-OF-THOUGHT ===")
print(f"Accuracy: {cot_acc:.2f} ({int(cot_acc*5)}/5)  Tokens: {cot_tokens}  Latency: {cot_latency:.1f}ms")
for true_a, pred_a, ok in cot_results:
    print(f"  [{'OK' if ok else 'WRONG'}] true={true_a} pred={pred_a}")

print(f"\nDelta: accuracy {cot_acc-direct_acc:+.2f}, tokens {cot_tokens-direct_tokens:+d} ({(cot_tokens/direct_tokens-1)*100:+.1f}%), latency {cot_latency-direct_latency:+.1f}ms ({(cot_latency/direct_latency-1)*100:+.1f}%)")

=== DIRECT ANSWER ===
Accuracy: 0.00 (0/5)  Tokens: 367  Latency: 4704.9ms
  [WRONG] true=12.0 pred=8.0
  [WRONG] true=210.0 pred=135.0
  [WRONG] true=33.0 pred=41.0
  [WRONG] true=47.0 pred=10.0
  [WRONG] true=2.2 pred=1.47

=== CHAIN-OF-THOUGHT ===
Accuracy: 1.00 (5/5)  Tokens: 1844  Latency: 18802.8ms
  [OK] true=12.0 pred=12.0
  [OK] true=210.0 pred=210.0
  [OK] true=33.0 pred=33.0
  [OK] true=47.0 pred=47.0
  [OK] true=2.2 pred=2.2

Delta: accuracy +1.00, tokens +1477 (+402.5%), latency +14097.9ms (+299.6%)


### Output Explanation: Direct-Answer vs. Chain-of-Thought

A stark, real result: direct-answer scored `0.00 (0/5)` — it got **every single problem wrong** (e.g. the sugar-cost problem: true `2.2`, predicted `1.47`; the train problem: true `210.0`, predicted `135.0`). Chain-of-Thought scored a perfect `1.00 (5/5)`, matching every true answer exactly. This is not a marginal improvement — on this real, genuinely multi-step problem set, direct-answer prompting failed completely while CoT succeeded completely, real, measured confirmation of Module 02's core claim that explicit reasoning space matters specifically for problems with genuine intermediate structure.

That real accuracy gain came at a real, substantial cost: CoT used `1844` total tokens vs. direct-answer's `367` — a real `+402.5%` increase — and took `18802.8ms` vs. `4704.9ms`, a real `+299.6%` latency increase. Unlike Notebook 01's few-shot result (where the cost bought nothing), here the cost bought everything — a real illustration that the accuracy-vs-cost trade-off genuinely depends on the task, exactly as Module 02 frames it: no technique is assumed to win, and this specific task set happens to make CoT's cost obviously worth paying.

## 2. Self-Consistency: Real Empirical Majority Vote vs. Theoretical Formula

15 real, independent samples at $T=0.7$ on the hardest problem (the fractional sugar-cost problem). Non-overlapping partitions of these SAME 15 real draws give real empirical estimates at k=1 (all 15 individually), k=3 (5 real groups of 3), and k=5 (3 real groups of 5) -- reusing one real sample pool efficiently rather than drawing fresh calls per k.

In [3]:
HARD_PROBLEM, HARD_ANSWER = EVAL_PROBLEMS[4]
print(f"Hard problem: {HARD_PROBLEM}")
print(f"True answer: {HARD_ANSWER}")

def sample_once():
    resp = client.chat.completions.create(
        model=MODEL, temperature=0.7, max_tokens=400,
        messages=[{"role": "system", "content": COT_SYSTEM}, {"role": "user", "content": HARD_PROBLEM}],
    )
    content = resp.choices[0].message.content
    final_line = content.split("Final Answer:")[-1] if "Final Answer:" in content else content
    return parse_number(final_line)

N = 15
samples = [sample_once() for _ in range(N)]
print(f"\nReal 15 samples: {samples}")

def is_close(x, true_val=HARD_ANSWER):
    return x is not None and abs(x - true_val) < 0.01

def majority_answer(group):
    from collections import Counter
    counts = Counter(group)
    return counts.most_common(1)[0][0]

# k=1: every individual sample is its own trial
p_hat = sum(is_close(s) for s in samples) / N

# k=3: 5 non-overlapping real groups of 3
groups_k3 = [samples[i:i+3] for i in range(0, 15, 3)]
k3_correct = sum(is_close(majority_answer(g)) for g in groups_k3)
k3_empirical = k3_correct / len(groups_k3)

# k=5: 3 non-overlapping real groups of 5
groups_k5 = [samples[i:i+5] for i in range(0, 15, 5)]
k5_correct = sum(is_close(majority_answer(g)) for g in groups_k5)
k5_empirical = k5_correct / len(groups_k5)

def majority_vote_probability(p, k):
    threshold = k // 2 + 1
    return sum(math.comb(k, i) * (p**i) * ((1-p)**(k-i)) for i in range(threshold, k+1))

print(f"\nReal empirical p_hat (k=1, single-sample accuracy over {N} real draws): {p_hat:.3f}")
print(f"Real empirical k=3 majority-vote accuracy ({len(groups_k3)} real groups): {k3_empirical:.3f} ({k3_correct}/{len(groups_k3)})")
print(f"Real empirical k=5 majority-vote accuracy ({len(groups_k5)} real groups): {k5_empirical:.3f} ({k5_correct}/{len(groups_k5)})")

theory_k3 = majority_vote_probability(p_hat, 3)
theory_k5 = majority_vote_probability(p_hat, 5)
print(f"\nTheoretical formula prediction at measured p_hat={p_hat:.3f}: k=3 -> {theory_k3:.3f}, k=5 -> {theory_k5:.3f}")
print(f"Real vs theoretical gap: k=3 {k3_empirical-theory_k3:+.3f}, k=5 {k5_empirical-theory_k5:+.3f}")
print("\nNote: real k=3/k=5 estimates come from only 5/3 real groups each -- a small real sample, reported as-is, not smoothed.")

Hard problem: A recipe requires 3/4 cup of sugar for 12 cookies. If you want to make 44 cookies, and each cup of sugar costs $0.80, what is the total cost of sugar needed in dollars (round to the nearest cent)?
True answer: 2.2



Real 15 samples: [2.2, 2.2, 5.0, 2.2, 2.2, 2.2, 2.2, 2.2, 2.2, 2.2, 2.2, 2.2, 2.2, 2.2, 2.2]

Real empirical p_hat (k=1, single-sample accuracy over 15 real draws): 0.933
Real empirical k=3 majority-vote accuracy (5 real groups): 1.000 (5/5)
Real empirical k=5 majority-vote accuracy (3 real groups): 1.000 (3/3)

Theoretical formula prediction at measured p_hat=0.933: k=3 -> 0.987, k=5 -> 0.997
Real vs theoretical gap: k=3 +0.013, k=5 +0.003

Note: real k=3/k=5 estimates come from only 5/3 real groups each -- a small real sample, reported as-is, not smoothed.


### Output Explanation: Self-Consistency — Empirical vs. Theoretical

The real 15 samples were `[2.2, 2.2, 5.0, 2.2, 2.2, 2.2, 2.2, 2.2, 2.2, 2.2, 2.2, 2.2, 2.2, 2.2, 2.2]` — only one real outlier (`5.0`, likely a real arithmetic slip on the fraction step) among 15 draws, giving a real, already-high `p_hat=0.933`. Both real empirical majority-vote estimates came out at a perfect `1.000` — `5/5` real groups at k=3 and `3/3` real groups at k=5 — because the single outlier only ever appeared alongside two or four correct answers in any group it landed in, so majority vote always outvoted it.

Compared against the theoretical formula evaluated at this same measured `p_hat=0.933`: the formula predicts `0.987` at k=3 and `0.997` at k=5 — both real empirical values (`1.000`) came out real `+0.013` and `+0.003` *above* the theoretical prediction. This is the opposite direction from Module 02's stated caveat (real correlated samples were expected to make empirical gains *smaller* than the idealized formula, not larger) — but the honest explanation is sample size, not a contradiction of the theory: with only `5` and `3` real groups respectively, a single outlier landing differently would have shifted the empirical rate substantially, and `1.000` is exactly what a small real sample with one rare outlier produces regardless of which direction the true gap runs. The more informative real finding here is that `p_hat=0.933` was already high — self-consistency had very little real headroom to demonstrate a large gain on a problem the model mostly gets right on a single sample anyway; the technique's real value is greater on harder problems with a lower single-sample `p`.

## 3. Real k-Sample Latency Multiplier: Sequential vs. Parallel

Real wall-clock timing: $k=1$ single call vs. $k=5$ parallel real calls (`ThreadPoolExecutor`), same simple fixed prompt, mirroring `04_ai_agents_and_protocols` Module 02's real parallel-call methodology.

In [4]:
LATENCY_PROBLEM = EVAL_PROBLEMS[1][0]  # the train-speed problem

def single_call():
    resp = client.chat.completions.create(
        model=MODEL, temperature=0.7, max_tokens=400,
        messages=[{"role": "system", "content": COT_SYSTEM}, {"role": "user", "content": LATENCY_PROBLEM}],
    )
    return resp.choices[0].message.content

start_k1 = time.perf_counter()
single_call()
k1_latency_ms = (time.perf_counter() - start_k1) * 1000

start_k5 = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as executor:
    list(executor.map(lambda _: single_call(), range(5)))
k5_latency_ms = (time.perf_counter() - start_k5) * 1000

print(f"Real k=1 (single call) latency: {k1_latency_ms:.1f}ms")
print(f"Real k=5 (5 parallel calls) latency: {k5_latency_ms:.1f}ms")
print(f"Real ratio: {k5_latency_ms/k1_latency_ms:.2f}x (NOT 5x, since the 5 calls run concurrently, not sequentially)")

Real k=1 (single call) latency: 2709.3ms
Real k=5 (5 parallel calls) latency: 3963.0ms
Real ratio: 1.46x (NOT 5x, since the 5 calls run concurrently, not sequentially)


### Output Explanation: Real k-Sample Latency Multiplier

The real single call took `2709.3ms`; the real 5 parallel calls took `3963.0ms` — a real ratio of `1.46x`, nowhere close to `5x`. This directly confirms Module 02's claim: since the 5 self-consistency samples have no dependency on each other, running them concurrently via `ThreadPoolExecutor` means wall-clock time is bounded by the slowest individual call plus real concurrency overhead, not the sum of all 5 sequential calls. The real `1.46x` (not `1.0x`) reflects genuine overhead from 5 simultaneous real network requests contending for resources — a real, honest measurement rather than the idealized "same as 1 call" outcome a purely theoretical parallel-speedup argument might suggest.

## 4. Cleanup

In [5]:
del client
print("Real OpenAI client released. This notebook used no local GPU model, so no CUDA cleanup is needed.")

Real OpenAI client released. This notebook used no local GPU model, so no CUDA cleanup is needed.
